# Twenty Questions Experiment Analysis

This notebook analyzes results from experiments produced by `fixed_game_loop.py`.

**Input:** `run_dir` - a directory in `experiments/` (e.g., `experiments/run_2025_11_17_210902`)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Enable high-DPI (Retina) rendering for inline Matplotlib
%config InlineBackend.figure_format = 'retina'

## 1. Configuration

Set the `run_dir` to the experiment directory you want to analyze:

In [ ]:
# Set the experiment directory to analyze
run_dir = (
    # "experiments/run_2025_11_18_002020"
    # "experiments/run_2025_11_18_153511"
    "experiments/run_2025_11_18_161752"
)

# Verify directory exists
if not os.path.exists(run_dir):
    raise ValueError(f"Directory {run_dir} does not exist")

print(f"Analyzing experiment: {run_dir}")

## 2. Load Data

In [ ]:
# Load summary.json
summary_path = os.path.join(run_dir, 'summary.json')
with open(summary_path, 'r') as f:
    summary_data = json.load(f)

# Extract experiment metadata
experiment_meta = summary_data['experiment_summary']
print("\n=== Experiment Metadata ===")
print(f"Total games: {experiment_meta['total_games']}")
print(f"Completed games: {experiment_meta['completed_games']}")
print(f"Failed games: {experiment_meta['failed_games']}")
print(f"Models tested: {experiment_meta['models_tested']}")
print(f"Agent types tested: {experiment_meta['agent_types_tested']}")
print(f"Games per model per agent: {experiment_meta['games_per_model_per_agent']}")
print(f"Total duration: {experiment_meta['total_duration']:.1f} seconds ({experiment_meta['total_duration']/60:.1f} minutes)")

In [ ]:
# Create summary DataFrame from summary.json results
summary_df = pd.DataFrame(summary_data['results'])
# Ensure agent_type ordering in summary_df for consistency
agent_order = ['LLM', 'Bayes-Q', 'Bayes-M', 'Bayes-QM']
if 'agent_type' in summary_df.columns:
    summary_df['agent_type'] = pd.Categorical(summary_df['agent_type'], categories=agent_order, ordered=True)
print(f"\nSummary DataFrame shape: {summary_df.shape}")
summary_df.head()

In [ ]:
# Load all individual game JSONs from games/ subdirectory
games_dir = os.path.join(run_dir, 'games')
game_files = list(Path(games_dir).glob('*.json'))

print(f"Found {len(game_files)} game files")

# Load all games into a list
games = []
for game_file in game_files:
    with open(game_file, 'r') as f:
        games.append(json.load(f))

# Create games DataFrame
game_df = pd.DataFrame(games)
# Ensure agent_type ordering in game_df for consistency
agent_order = ['LLM', 'Bayes-Q', 'Bayes-M', 'Bayes-QM']
if 'agent_type' in game_df.columns:
    game_df['agent_type'] = pd.Categorical(game_df['agent_type'], categories=agent_order, ordered=True)
print(f"\nGames DataFrame shape: {game_df.shape}")
game_df.head()

## 3. Data Processing

Extract key metrics from the game data:

In [ ]:
# Filter out failed games (games with 'error' field)
successful_games = game_df[~game_df['error'].notna()].copy() if 'error' in game_df.columns else game_df.copy()

# Enforce consistent ordering for agent types used throughout the notebook
agent_order = ['LLM', 'Bayes-Q', 'Bayes-M', 'Bayes-QM']
if 'agent_type' in successful_games.columns:
    successful_games['agent_type'] = pd.Categorical(successful_games['agent_type'], categories=agent_order, ordered=True)

print(f"Successful games: {len(successful_games)} / {len(game_df)}")

# Extract win status from rewards (reward of 1 means win)
successful_games['won'] = successful_games['rewards'].apply(lambda x: x.get('0', 0) == 1 if isinstance(x, dict) else False)

# Extract game reason
successful_games['outcome_reason'] = successful_games['game_info'].apply(
    lambda x: x.get('0', {}).get('reason', 'Unknown') if isinstance(x, dict) else 'Unknown'
)

print(f"\nWin rate: {successful_games['won'].mean():.2%}")
print(f"Average turns: {successful_games['turn_count'].mean():.2f}")
print(f"Average duration: {successful_games['game_duration'].mean():.2f} seconds")

## 4. Basic Analysis

### 4.1 Win Rates by Agent Type and Model

In [ ]:
# Win rates by agent type (preserve specified agent ordering)
win_rate_by_agent = successful_games.groupby('agent_type')['won'].agg(['mean', 'count', 'sum'])
win_rate_by_agent.columns = ['win_rate', 'total_games', 'wins']
# Reindex to fixed agent order instead of sorting by win rate
agent_order = ['LLM', 'Bayes-Q', 'Bayes-M', 'Bayes-QM']
win_rate_by_agent = win_rate_by_agent.reindex(agent_order)

print("\n=== Win Rates by Agent Type ===")
print(win_rate_by_agent)

# Win rates by model
win_rate_by_model = successful_games.groupby('model_name')['won'].agg(['mean', 'count', 'sum'])
win_rate_by_model.columns = ['win_rate', 'total_games', 'wins']
win_rate_by_model = win_rate_by_model.sort_values('win_rate', ascending=False)

print("\n=== Win Rates by Model ===")
print(win_rate_by_model)

# Win rates by both agent type and model
win_rate_by_both = successful_games.groupby(['model_name', 'agent_type'])['won'].agg(['mean', 'count', 'sum'])
win_rate_by_both.columns = ['win_rate', 'total_games', 'wins']
win_rate_by_both = win_rate_by_both.sort_values('win_rate', ascending=False)

print("\n=== Win Rates by Model and Agent Type ===")
print(win_rate_by_both)

In [ ]:
win_rate_by_agent

In [ ]:
# Visualize win rates
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Win rate by agent type
ax1 = axes[0]
# Ensure plotting follows fixed agent order
win_rate_by_agent = win_rate_by_agent.reindex(['LLM','Bayes-Q','Bayes-M','Bayes-QM'])
win_rate_by_agent['win_rate'].plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Win Rate by Agent Type', fontsize=14, fontweight='bold')
ax1.set_xlabel('Agent Type', fontsize=12)
ax1.set_ylabel('Win Rate', fontsize=12)
ax1.set_ylim(0, 1)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels on bars
for i, v in enumerate(win_rate_by_agent['win_rate']):
    ax1.text(i, v + 0.02, f'{v:.1%}', ha='center', va='bottom', fontweight='bold')

# Win rate by model
ax2 = axes[1]
win_rate_by_model['win_rate'].plot(kind='bar', ax=ax2, color='coral')
ax2.set_title('Win Rate by Model', fontsize=14, fontweight='bold')
ax2.set_xlabel('Model', fontsize=12)
ax2.set_ylabel('Win Rate', fontsize=12)
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels on bars
for i, v in enumerate(win_rate_by_model['win_rate']):
    ax2.text(i, v + 0.02, f'{v:.1%}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart: Win rate by model and agent type
win_rate_pivot = successful_games.pivot_table(
    values='won',
    index='model_name',
    columns='agent_type',
    aggfunc='mean',
)
# Ensure pivot columns are in the requested agent order
agent_order = ['LLM', 'Bayes-Q', 'Bayes-M', 'Bayes-QM']
win_rate_pivot = win_rate_pivot.reindex(columns=agent_order)

ax = win_rate_pivot.plot(kind='bar', figsize=(12, 6), width=0.8)
ax.set_title('Win Rate by Model and Agent Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Win Rate', fontsize=12)
ax.set_ylim(0, 1)
ax.legend(title='Agent Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4.2 Number of Turns by Agent Type and Model

In [ ]:
# Average turns by agent type
turns_by_agent = successful_games.groupby('agent_type')['turn_count'].agg(['mean', 'std', 'count'])
turns_by_agent.columns = ['avg_turns', 'std_turns', 'count']
turns_by_agent = turns_by_agent.sort_values('avg_turns')

print("\n=== Average Turns by Agent Type ===")
print(turns_by_agent)

# Average turns by model
turns_by_model = successful_games.groupby('model_name')['turn_count'].agg(['mean', 'std', 'count'])
turns_by_model.columns = ['avg_turns', 'std_turns', 'count']
turns_by_model = turns_by_model.sort_values('avg_turns')

print("\n=== Average Turns by Model ===")
print(turns_by_model)

# Average turns by both agent type and model
turns_by_both = successful_games.groupby(['model_name', 'agent_type'])['turn_count'].agg(['mean', 'std', 'count'])
turns_by_both.columns = ['avg_turns', 'std_turns', 'count']
turns_by_both = turns_by_both.sort_values('avg_turns')

print("\n=== Average Turns by Model and Agent Type ===")
print(turns_by_both)

In [ ]:
# Visualize turn counts
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Average turns by agent type
ax1 = axes[0]
turns_by_agent['avg_turns'].plot(kind='bar', ax=ax1, color='steelblue', yerr=turns_by_agent['std_turns'])
ax1.set_title('Average Turns by Agent Type', fontsize=14, fontweight='bold')
ax1.set_xlabel('Agent Type', fontsize=12)
ax1.set_ylabel('Average Turns', fontsize=12)
ax1.grid(axis='y', alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels on bars
for i, v in enumerate(turns_by_agent['avg_turns']):
    ax1.text(i, v + 0.5, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

# Average turns by model
ax2 = axes[1]
turns_by_model['avg_turns'].plot(kind='bar', ax=ax2, color='coral', yerr=turns_by_model['std_turns'])
ax2.set_title('Average Turns by Model', fontsize=14, fontweight='bold')
ax2.set_xlabel('Model', fontsize=12)
ax2.set_ylabel('Average Turns', fontsize=12)
ax2.grid(axis='y', alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add value labels on bars
for i, v in enumerate(turns_by_model['avg_turns']):
    ax2.text(i, v + 0.5, f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart: Average turns by model and agent type
turns_pivot = successful_games.pivot_table(
    values='turn_count',
    index='model_name',
    columns='agent_type',
    aggfunc='mean',
)
# Ensure pivot columns are in the requested agent order
agent_order = ['LLM', 'Bayes-Q', 'Bayes-M', 'Bayes-QM']
turns_pivot = turns_pivot.reindex(columns=agent_order)

ax = turns_pivot.plot(kind='bar', figsize=(12, 6), width=0.8)
ax.set_title('Average Turns by Model and Agent Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Average Turns', fontsize=12)
ax.legend(title='Agent Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Box plot: Turn count distribution by agent type
fig, ax = plt.subplots(figsize=(12, 6))
successful_games.boxplot(column='turn_count', by='agent_type', ax=ax)
ax.set_title('Turn Count Distribution by Agent Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Agent Type', fontsize=12)
ax.set_ylabel('Turn Count', fontsize=12)
plt.suptitle('')  # Remove the default title
plt.tight_layout()
plt.show()

### 4.3 Additional Analysis: Win Rate vs Turn Count

In [ ]:
# Analyze relationship between turns and winning
print("\n=== Turns Analysis by Outcome ===")
print("\nWinning games:")
print(successful_games[successful_games['won']]['turn_count'].describe())
print("\nLosing games:")
print(successful_games[~successful_games['won']]['turn_count'].describe())

In [ ]:
# Distribution of turn counts for wins vs losses
fig, ax = plt.subplots(figsize=(12, 6))

successful_games[successful_games['won']]['turn_count'].hist(
    bins=20, alpha=0.6, label='Wins', ax=ax, color='green'
)
successful_games[~successful_games['won']]['turn_count'].hist(
    bins=20, alpha=0.6, label='Losses', ax=ax, color='red'
)

ax.set_title('Turn Count Distribution: Wins vs Losses', fontsize=14, fontweight='bold')
ax.set_xlabel('Turn Count', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 4.4 Game Duration Analysis

In [ ]:
# Average duration by agent type and model
duration_by_agent = successful_games.groupby('agent_type')['game_duration'].agg(['mean', 'std', 'count'])
duration_by_agent.columns = ['avg_duration', 'std_duration', 'count']
duration_by_agent = duration_by_agent.sort_values('avg_duration')

print("\n=== Average Duration (seconds) by Agent Type ===")
print(duration_by_agent)

duration_by_model = successful_games.groupby('model_name')['game_duration'].agg(['mean', 'std', 'count'])
duration_by_model.columns = ['avg_duration', 'std_duration', 'count']
duration_by_model = duration_by_model.sort_values('avg_duration')

print("\n=== Average Duration (seconds) by Model ===")
print(duration_by_model)

## 5. Summary Statistics Table

In [ ]:
# Create comprehensive summary table
summary_stats = successful_games.groupby(['model_name', 'agent_type']).agg({
    'won': ['mean', 'sum', 'count'],
    'turn_count': ['mean', 'std', 'min', 'max'],
    'game_duration': ['mean', 'std']
}).round(2)

summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]
summary_stats = summary_stats.rename(columns={
    'won_mean': 'win_rate',
    'won_sum': 'wins',
    'won_count': 'total_games',
    'turn_count_mean': 'avg_turns',
    'turn_count_std': 'std_turns',
    'turn_count_min': 'min_turns',
    'turn_count_max': 'max_turns',
    'game_duration_mean': 'avg_duration_sec',
    'game_duration_std': 'std_duration_sec'
})

print("\n=== Comprehensive Summary Statistics ===")
print(summary_stats.sort_values('win_rate', ascending=False))

In [ ]:
# Export summary to CSV
output_csv = os.path.join(run_dir, 'analysis_summary.csv')
summary_stats.to_csv(output_csv)
print(f"\nSummary statistics exported to: {output_csv}")

# Entropy over turns by agent type
We aggregate per-turn entropy from the `diagnostics` field of `summary_df` and plot mean entropy over turns with confidence intervals, grouped by `agent_type`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Expect: summary_df with columns including 'agent_type' and 'diagnostics'
assert 'summary_df' in globals(), "summary_df must be defined earlier in the notebook."
assert 'diagnostics' in summary_df.columns, "summary_df must include a 'diagnostics' column."

rows = []
for idx, row in summary_df.iterrows():
    agent = row.get('agent_type', 'Unknown')
    game_id = row.get('game_id', idx)  # fall back to row index if no explicit id
    run_id = row.get('run_id', row.get('run_id', 'unknown'))  # handle absence
    diag = row.get('diagnostics', None)
    if diag is None:
        continue
    # If diagnostics comes as JSON string, parse it
    if isinstance(diag, str):
        try:
            diag = json.loads(diag)
        except Exception:
            diag = None
    if diag is None:
        continue

    entropies = None

    # Common patterns: direct list, list of dicts, dict with 'entropy' or 'turns'
    if isinstance(diag, dict):
        if 'entropy' in diag:
            ent = diag['entropy']
            if isinstance(ent, list):
                if len(ent) and isinstance(ent[0], dict) and 'entropy' in ent[0]:
                    entropies = [e.get('entropy') for e in ent if e is not None and 'entropy' in e]
                else:
                    entropies = ent
            elif isinstance(ent, (int, float)):
                entropies = [float(ent)]
        elif 'turns' in diag and isinstance(diag['turns'], list):
            entropies = [t.get('entropy') for t in diag['turns'] if isinstance(t, dict) and 'entropy' in t]
    elif isinstance(diag, list):
        # maybe list of per-turn dicts with entropy
        if len(diag) and isinstance(diag[0], dict) and 'entropy' in diag[0]:
            entropies = [d.get('entropy') for d in diag if d is not None and 'entropy' in d]
        elif len(diag) and isinstance(diag[0], (int, float)):
            entropies = diag

    if not entropies:
        continue

    # Normalize to floats, skip Nones
    entropies = [float(e) for e in entropies if e is not None and not (isinstance(e, float) and (np.isnan(e) or np.isinf(e)))]

    for t, e in enumerate(entropies, start=1):
        rows.append({
            'agent_type': agent,
            'game_id': game_id,
            'run_id': run_id,
            'turn': t,
            'entropy': e,
        })

entropy_df = pd.DataFrame(rows)

if entropy_df.empty:
    print("No per-turn entropy data found in summary_df['diagnostics'].")
else:
    # Order agent types if available
    if 'agent_type' in summary_df.columns and isinstance(summary_df['agent_type'].dtype, pd.CategoricalDtype):
        order = list(summary_df['agent_type'].cat.categories)
        entropy_df['agent_type'] = pd.Categorical(entropy_df['agent_type'], categories=order, ordered=True)

    # Primary: mean with CI by agent type
    plt.figure(figsize=(9, 5))
    sns.lineplot(
        data=entropy_df,
        x='turn', y='entropy', hue='agent_type',
        estimator='mean', errorbar='ci', linewidth=2
    )
    plt.title('Entropy Over Turns by Agent Type', fontsize=14, fontweight='bold')
    plt.xlabel('Turn')
    plt.ylabel('Entropy (bits)')
    plt.grid(True, axis='y', alpha=0.25)
    plt.legend(title='Agent Type', frameon=False)
    plt.tight_layout()
    plt.show()

    # Secondary: per-game trajectories colored by run_id with numeric legend ordering
    entropy_df['run_id'] = entropy_df['run_id'].astype(str)

    def _run_sort_key(r):
        try:
            return (0, int(r))
        except ValueError:
            return (1, r)

    sorted_run_ids = sorted(entropy_df['run_id'].unique(), key=_run_sort_key)
    palette = sns.color_palette('tab20', n_colors=len(sorted_run_ids)) if len(sorted_run_ids) <= 20 else None

    g = sns.relplot(
        data=entropy_df,
        x='turn', y='entropy', col='agent_type', col_wrap=4,
        kind='line', estimator=None, units='game_id', hue='run_id', hue_order=sorted_run_ids,
        alpha=0.5, height=3.0,
        facet_kws=dict(sharey=True), palette=palette, legend='brief'
    )
    g.set_axis_labels('Turn', 'Entropy (bits)')
    g.set_titles(col_template='{col_name}')
    for ax in g.axes.flat:
        ax.grid(True, axis='y', alpha=0.2)
    g.figure.suptitle('Per-Game Entropy Trajectories by Agent Type (Colored by run_id)', y=1.02, fontsize=13, fontweight='bold')
    g.tight_layout()


# Probability of Ground Truth Over Turns by Agent Type
We extract the model's assigned probability to the actual `ground_truth_word` at each turn (from `diagnostics['turns'][i]['belief_state_before_question']`) and plot its evolution. Higher curves indicate faster convergence on the true answer.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure raw game JSONs are loaded (games list from earlier cell)
assert 'games' in globals(), "List 'games' with raw game JSON objects must be defined earlier."

prob_rows = []
for g in games:
    agent = g.get('agent_type', 'Unknown')
    gt_word = g.get('ground_truth_word')
    game_id = g.get('game_id', None)
    run_id = g.get('run_id', 'unknown')
    diagnostics = g.get('diagnostics', {})
    turns = diagnostics.get('turns', []) if isinstance(diagnostics, dict) else []
    if not gt_word or not turns:
        continue
    for turn_entry in turns:
        turn_num = turn_entry.get('turn_number')
        belief = turn_entry.get('belief_state_before_question', {})
        if not isinstance(belief, dict):
            continue
        p_true = belief[gt_word.lower()]
        prob_rows.append({
            'agent_type': agent,
            'game_id': game_id,
            'run_id': run_id,
            'ground_truth_word': gt_word,
            'turn': int(turn_num) if turn_num is not None else len(prob_rows),
            'p_true': float(p_true)
        })

p_true_df = pd.DataFrame(prob_rows)

if p_true_df.empty:
    print("No per-turn ground truth probabilities found.")
else:
    # Preserve agent ordering
    if 'agent_type' in p_true_df.columns and 'agent_type' in summary_df.columns and isinstance(summary_df['agent_type'].dtype, pd.CategoricalDtype):
        order = list(summary_df['agent_type'].cat.categories)
        p_true_df['agent_type'] = pd.Categorical(p_true_df['agent_type'], categories=order, ordered=True)

    # Treat run_id as categorical for coloring & sort numerically where possible
    p_true_df['run_id'] = p_true_df['run_id'].astype(str)
    def _run_sort_key(r):
        try:
            return (0, int(r))
        except ValueError:
            return (1, r)
    sorted_run_ids = sorted(p_true_df['run_id'].unique(), key=_run_sort_key)
    palette = sns.color_palette('tab20', n_colors=len(sorted_run_ids)) if len(sorted_run_ids) <= 20 else None

    # Aggregate mean with 95% CI (still across run_ids)
    plt.figure(figsize=(9,5))
    sns.lineplot(
        data=p_true_df,
        x='turn', y='p_true', hue='agent_type',
        estimator='mean', errorbar='ci', linewidth=2
    )
    plt.title('Probability of Ground Truth Word Over Turns', fontsize=14, fontweight='bold')
    plt.xlabel('Turn')
    plt.ylabel('P(True Word)')
    plt.ylim(0,1)
    plt.grid(True, axis='y', alpha=0.25)
    plt.legend(title='Agent Type', frameon=False)
    plt.tight_layout()
    plt.show()

    # Faceted per-game trajectories colored by run_id (sorted legend)
    g = sns.relplot(
        data=p_true_df,
        x='turn', y='p_true', col='agent_type', col_wrap=4,
        kind='line', estimator=None, units='game_id', hue='run_id', hue_order=sorted_run_ids,
        alpha=0.45, height=3.0,
        facet_kws=dict(sharey=True), palette=palette, legend='brief'
    )
    g.set_axis_labels('Turn', 'P(True Word)')
    g.set_titles(col_template='{col_name}')
    for ax in g.axes.flat:
        ax.grid(True, axis='y', alpha=0.2)
        ax.set_ylim(0,1)
    g.figure.suptitle('Per-Game Trajectories of Ground Truth Probability (Colored by run_id)', y=1.02, fontsize=13, fontweight='bold')
    g.tight_layout()

    # AUC summary (sum of probabilities across turns)
    auc_df = p_true_df.groupby(['game_id','agent_type'])[['turn','p_true']].apply(lambda df: (df.sort_values('turn')['p_true'].sum()))
    auc_df = auc_df.reset_index(name='auc_sum')
    auc_summary = auc_df.groupby('agent_type')['auc_sum'].agg(['mean','std','count']).round(3)
    print('\n=== AUC (Sum of P(True Word) Across Turns) by Agent Type ===')
    print(auc_summary)


In [ ]:
p_true_df.run_id.unique()

In [ ]:
game_df.run_id.unique()